# TM-RugPull dataset initial analysis

## Data collection

In [1]:
#Loading data from .xlsx file

import pandas as pd
import numpy as np
import matplotlib.pyplot as pyplot

file = 'data/TM-RugPull.xlsx'

data = pd.read_excel(file)

#Remove a space in the end of some column names
data.columns = data.columns.str.strip()

print(data.shape)

pd.set_option('display.max_columns', None)

data.head(5)

(1000, 27)


,Project Title,MaxPrice (Quarter 1),MaxPrice (Quarter 2),MaxPrice (Quarter 3),MaxPrice (Quarter 4),Blockchain,the number of Transactions,Token concentration ratio per holder,Total Variance,Variance of holders with more than 1% tokens,Token balance,Sign,first deposits,Blockchain Type,Smart Contract (online),smart Contract (offline),website,x profile,class,project starting date,project end date,Google results for project title (first day),Google results for project title (project duration/2),Google results for project website (first day),Google results for project website (duration/2),Google results for project x profile (first days),Google results for project x profile (duration/2)
0,HyperVerse Token (HVT),7.650000e+00,1.500000e-01,9.100000e-06,8.000000e-07,BSC,623909,25752,9.537265e-03,4.929203e-02,166600000000100,HVT,44,POSA,sourse code,CODE,https://thehyperverse.net/index.html,https://twitter.com/HyperVerse6,scam,2022-01-27,2023-07-14,148,148,29,8,74,47
1,Fintoch,1.795000e-11,1.907000e-11,1.727000e-10,1.769000e-10,BSC,"1,492,842","147,791",2.487228e+03,4.503032e+07,34403.74594,BEP-20 TOKEN*,1,POSA,sourse code,CODE,https://web.archive.org/web/20230603123631/htt...,NaN,scam,2022-07-12,2023-06-19,820,167,4530,1590,48,4
2,Flare Token,1.955000e-03,5.519000e-04,4.158000e-04,2.838000e-04,BSC,"184,694",15173,8.901714e+14,6.695544e+17,10000000000,Flare,53,POSA,sourse code,CODE,https://pipeflare.io/,https://x.com/MetaFlareToken,scam,2021-10-24,2022-11-24,421,156,6,5,1710,1150
3,Safuu Protocol,2.070000e+02,2.110000e+02,7.000000e+01,2.400000e+01,BSC,"275,530",151979,2.457331e+10,8.139816e+14,61634066.59803,SAFUU,2,POSA,sourse code,CODE,https://safuu.com/,https://x.com/safuuxofficial,scam,2022-02-03,2022-08-13,327,323,0,0,5,4
4,SCT,2.986000e-01,1.659000e-01,1.831000e-01,1.552000e-01,BSC,8445,"6,126\n",4.252172e+06,1.410127e+05,"42,896,736.739367",SCT,52,POSA,sourse code,CODE,https://supercells.jp/en/,https://x.com/scttoken,scam,2023-02-27,2024-07-09,4990000,345000,6,3,1830,594


##### Create a test set and a validation test

In [2]:
#Create a test set and a validation set from the raw data to avoid data leakage
#Validation set size is about 10,5% and test set size is about 24,5% from the whole data set
#TODO reference to AML tutorial

from sklearn.model_selection import train_test_split

#define size of data both for test and validatioin sets, define the seed for all subsequent experiments
test_and_val_size = 0.35
seed = 7

#Split the data first on train set and set for test and validation
train_set, test_and_val_set = train_test_split(data, test_size=test_and_val_size, random_state=seed, stratify=data['class'])

#Split the part for test and validation into test set and validation set
test_set, val_set = train_test_split(test_and_val_set, test_size=0.3, random_state=seed, stratify=test_and_val_set['class'])

#Output the shapes to verify the splits
print("Training set shape:", train_set.shape)
print("Test set shape:", test_set.shape)
print("Validation set shape:", val_set.shape)

#Create a list of sets to perform further feature engeneering on all subsets of data
data_sets = [train_set, test_set, val_set]

Training set shape: (650, 27)
Test set shape: (245, 27)
Validation set shape: (105, 27)


In [3]:
# Verify that there is no overlap between sets, no data leak at this stage
print("Overlap between train and test:", np.intersect1d(train_set.index, test_set.index).size)
print("Overlap between train and validation:", np.intersect1d(train_set.index, val_set.index).size)
print("Overlap between test and validation:", np.intersect1d(test_set.index, val_set.index).size)

Overlap between train and test: 0
Overlap between train and validation: 0
Overlap between test and validation: 0


## Data analysis

In [4]:
#Check general info about data

print("\nDataset information:")
data.info()


Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 27 columns):
 #   Column                                                 Non-Null Count  Dtype         
---  ------                                                 --------------  -----         
 0   Project Title                                          1000 non-null   object        
 1   MaxPrice (Quarter 1)                                   1000 non-null   float64       
 2   MaxPrice (Quarter 2)                                   1000 non-null   float64       
 3   MaxPrice (Quarter 3)                                   1000 non-null   float64       
 4   MaxPrice (Quarter 4)                                   1000 non-null   float64       
 5   Blockchain                                             1000 non-null   object        
 6   the number of Transactions                             1000 non-null   object        
 7   Token concentration ratio per holder            

In [5]:
#Deleting columns that are not useful for further analysis

for set in data_sets:
    set.drop(columns=['Project Title', 'Sign', 'website', 'x profile', 'Smart Contract (online)', 'smart Contract (offline)', 'project starting date', 'project end date'], inplace=True)

In [7]:
train_set.head(10)

,MaxPrice (Quarter 1),MaxPrice (Quarter 2),MaxPrice (Quarter 3),MaxPrice (Quarter 4),Blockchain,the number of Transactions,Token concentration ratio per holder,Total Variance,Variance of holders with more than 1% tokens,Token balance,first deposits,Blockchain Type,class,Google results for project title (first day),Google results for project title (project duration/2),Google results for project website (first day),Google results for project website (duration/2),Google results for project x profile (first days),Google results for project x profile (duration/2)
380,5.334000e-02,5.334000e-02,5.334000e-02,1.758000e-01,ETH,41,30,5.733989e+08,2.112925e+09,160000,0.05334,POS,scam,0,0,0,0,0,0
366,4.148000e-06,4.074000e-06,4.074000e-06,4.074000e-06,ETH,74,62,1.049362e+28,1.359328e+29,1000066666666660,0.000004,POS,scam,1,1,0,0,0,0
967,1.118000e+01,5.190000e+00,3.140000e+00,1.800000e+01,BSC,90329,2089,6.759598e+07,1.039946e+10,850000,11.18,POSA,normal,5,3,2,2,140,66
185,5.199000e-05,0.000000e+00,3.385000e-05,4.763000e-05,BSC,1383,1234,4.183023e+26,7.124242e+28,1000000000000000,0.000052,POSA,scam,2,0,1,0,3,2
400,2.282000e-03,8.994000e-05,8.424000e-05,6.630000e-05,POLYGON,2592,736,1.245419e+07,4.681463e+08,195379.0521,0.002282,POS,scam,867,672,0,0,5,31
585,1.180000e-06,1.960000e-06,2.080000e-06,2.710000e-06,ETH,39,23,1.222502e+24,2.256094e+24,10000000000000,0.000001,POS,scam,0,0,0,0,8,0
25,3.968000e-11,7.038000e-12,4.437000e-12,4.523000e-12,BSC,230,775,1.920780e+08,2.814857e+07,33333333343333298176,0.0,POSA,scam,0,0,1,1,1770,1400
887,2.710000e+00,6.080000e-01,9.800000e-01,8.700000e-01,ETH,2321904,99497,2.302581e+11,1.235555e+15,424999998.000001,0.4,POS,normal,539,144,6,80,436,56
933,1.400000e-01,7.600000e-02,2.600000e-02,5.100000e-02,ETH,106528,22908,4.905284e+12,4.885311e+15,1000000000,0.049,POS,normal,177,112,4,0,17500,4040
487,4.800000e-02,9.190000e-02,7.400000e-02,4.150000e-02,ETH,26448,5678,6.689802e+11,1.068798e+14,331614076.921422,0.0305,POS,scam,4,3,40,42,5,3
